In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/__results__.html
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/change_rates_paper1_lccn.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/keel_norm_map.pkl
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/__notebook__.ipynb
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/__output__.json
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/lccn_ablation_paper1_lccn.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/custom.css
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/val.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/train_ids.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/val_ids.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/train.csv
/kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits/test.csv
/kaggl

In [2]:
# ================================================================
#  CELL 0 — INSTALL
# ================================================================
import subprocess, sys, os, shutil
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q scikit-learn xgboost scipy')
sh('pip install -q transformers accelerate --upgrade')

for p in ("trainer.py", "/kaggle/working/trainer.py"):
    if os.path.exists(p): os.remove(p); print("removed stale", p)
shutil.rmtree("__pycache__", ignore_errors=True)
sys.modules.pop("trainer", None)

import torch, transformers
print("\npython", sys.version.split()[0], "| torch", torch.__version__,
      "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())


+ pip install -q scikit-learn xgboost scipy
+ pip install -q transformers accelerate --upgrade
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.8 MB/s eta 0:00:00

python 3.12.13 | torch 2.10.0+cu128 | transformers 5.14.1 | cuda True


In [3]:
%%writefile trainer.py
"""
trainer.py v1.3 — shared classification method for BOTH papers.

Written identically by CELL 0 of NB2_paper1 and NB2_paper2. Only the
arm list and the model list differ, in one cell.

Key properties, all of which came out of the audit:
  * reads the id-keyed splits produced by NB1; never re-splits
  * ONE declared training budget for every transformer, so a MuRIL number
    from one paper is comparable with the same number from the other
  * every model is run over SEEDS and reported as mean +/- std
  * macro-F1 computed longhand so it visibly matches equations (5)-(8)
  * paired bootstrap (10,000 resamples) and paired t-test, equations (9)-(10)
  * per-comment predictions saved for every model x arm
"""
TRAINER_VERSION = "1.4"

import os, gc, glob, json, hashlib, warnings, itertools
import numpy as np, pandas as pd
from collections import Counter
warnings.filterwarnings("ignore")

SEED   = 42
SEEDS  = (42, 43, 44)          # neural models; classical are deterministic
WORK   = "/kaggle/working"
LABELS = ["anger", "disgust", "happy", "sad"]

# ---- the single declared training budget for ALL transformers ----
BUDGET = dict(epochs=10, lr=2e-5, batch_size=32, max_len=96,
              warmup_ratio=0.1, weight_decay=0.01, patience=3)
# ---- deep models ----
DEEP = dict(epochs=8, lr=1e-3, batch_size=64, max_len=64,
            vocab=40000, emb=200, hid=128, dropout=0.3, patience=2)

os.makedirs(f"{WORK}/preds", exist_ok=True)

MODEL_CKPT = {
    "XLM-RoBERTa": "xlm-roberta-base",
    "MuRIL":       "google/muril-base-cased",
    "mBERT":       "bert-base-multilingual-cased",
    "IndicBERTv2": "ai4bharat/IndicBERTv2-MLM-only",
}

# ============================================================ metrics (5)-(8)
def per_class_prf(y_true, y_pred, classes):
    out = {}
    for c in classes:
        tp = int(np.sum((y_pred == c) & (y_true == c)))
        fp = int(np.sum((y_pred == c) & (y_true != c)))
        fn = int(np.sum((y_pred != c) & (y_true == c)))
        p = tp/(tp+fp) if tp+fp else 0.0                      # equation (5)
        r = tp/(tp+fn) if tp+fn else 0.0                      # equation (6)
        f = 2*p*r/(p+r) if p+r else 0.0                       # equation (7)
        out[c] = (p, r, f)
    return out

def macro_f1(y_true, y_pred, classes=LABELS):
    """Equation (8): unweighted mean of the per-class F1 scores."""
    return float(np.mean([v[2] for v in per_class_prf(y_true, y_pred, classes).values()]))

# ================================================ significance (9)-(10)
def paired_bootstrap(y_true, pred_a, pred_b, classes=LABELS, B=10000, seed=SEED):
    """Resample the common test set B times; delta_b = F1_b(b) - F1_b(a).
    Returns the observed difference, the 95% CI and a two-sided p-value."""
    rng = np.random.default_rng(seed); n = len(y_true)
    obs = macro_f1(y_true, pred_b, classes) - macro_f1(y_true, pred_a, classes)
    d = np.empty(B)
    for i in range(B):
        ix = rng.integers(0, n, n); yt = y_true[ix]
        d[i] = macro_f1(yt, pred_b[ix], classes) - macro_f1(yt, pred_a[ix], classes)
    lo, hi = np.percentile(d, [2.5, 97.5])
    if np.allclose(d, 0.0):
        # identical systems: every resample gives zero difference, so there is
        # no evidence of a difference at all. Without this guard the formula
        # below would report a small p for two systems that never disagree.
        return dict(observed=obs, mean=0.0, ci_low=0.0, ci_high=0.0, p_value=1.0)
    le = int(np.sum(d <= 0))
    p = min(2.0 * min(le+1, B-le+1) / (B+1), 1.0)     # +1 smoothing, never an exact 0
    return dict(observed=obs, mean=float(d.mean()), ci_low=float(lo),
                ci_high=float(hi), p_value=float(p))

def paired_ttest(y_true, pred_a, pred_b):
    from scipy import stats
    ca = (pred_a == y_true).astype(float); cb = (pred_b == y_true).astype(float)
    t, p = stats.ttest_rel(cb, ca)
    return dict(t=float(t), p_value=float(p),
                acc_a=float(ca.mean()), acc_b=float(cb.mean()), n=int(len(y_true)))

# ============================================================== features
def make_vectorizer():
    """Equation (4): word TF-IDF unioned with character n-grams. The character
    channel is what keeps classical baselines competitive on noisy code-mixed
    text, where the word channel is very sparse."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import make_union
    return make_union(
        TfidfVectorizer(analyzer="word",    ngram_range=(1,2), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=3, sublinear_tf=True))

def classical_models():
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.ensemble import RandomForestClassifier
    M = {
        "Logistic Regression": lambda: LogisticRegression(max_iter=2000, C=4.0,
                                                          class_weight="balanced"),
        "SVM":                 lambda: LinearSVC(C=0.5, class_weight="balanced"),
        "Naive Bayes":         lambda: MultinomialNB(alpha=0.3),
        "Random Forest":       lambda: RandomForestClassifier(n_estimators=400, n_jobs=-1,
                                                              random_state=SEED),
    }
    try:
        from xgboost import XGBClassifier
        M["XGBoost"] = lambda: XGBClassifier(n_estimators=500, max_depth=7, learning_rate=0.1,
                                             subsample=0.9, colsample_bytree=0.9,
                                             tree_method="hist", n_jobs=-1,
                                             random_state=SEED, verbosity=0)
    except ImportError:
        print("xgboost unavailable -- skipping")
    return M

# ============================================================== classical
def run_classical(name, tr, va, te, col, classes=LABELS):
    ctor = classical_models()[name]
    vec  = make_vectorizer()
    Xtr  = vec.fit_transform(pd.concat([tr[col], va[col]]))
    ytr  = pd.concat([tr["label"], va["label"]])
    Xte  = vec.transform(te[col])
    clf  = ctor()
    if name == "XGBoost":
        lut = {c: i for i, c in enumerate(classes)}
        clf.fit(Xtr, ytr.map(lut))
        pred = np.array(classes)[clf.predict(Xte)]
    else:
        clf.fit(Xtr, ytr); pred = clf.predict(Xte)
    return macro_f1(te["label"].values, pred, classes), pred, None

# ================================================================== deep
def run_deep(name, tr, va, te, col, classes=LABELS, seeds=SEEDS):
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    cnn = (name == "CNN-BiLSTM")

    class Net(nn.Module):
        def __init__(self, V, C):
            super().__init__()
            self.emb = nn.Embedding(V, DEEP["emb"], padding_idx=0)
            self.cnn = nn.Conv1d(DEEP["emb"], DEEP["emb"], 3, padding=1) if cnn else None
            self.rnn = nn.LSTM(DEEP["emb"], DEEP["hid"], batch_first=True, bidirectional=True)
            self.dp  = nn.Dropout(DEEP["dropout"])
            self.fc  = nn.Linear(DEEP["hid"]*2, C)
        def forward(self, x):
            e = self.emb(x)
            if self.cnn is not None: e = torch.relu(self.cnn(e.transpose(1,2))).transpose(1,2)
            h, _ = self.rnn(e)
            m = (x != 0).unsqueeze(-1).float()
            return self.fc(self.dp((h*m).sum(1) / m.sum(1).clamp(min=1)))

    freq = Counter(w for t in tr[col] for w in str(t).split())
    vocab = {w: i+2 for i, (w, _) in enumerate(freq.most_common(DEEP["vocab"]-2))}
    lut = {c: i for i, c in enumerate(classes)}
    def enc(texts):
        a = np.zeros((len(texts), DEEP["max_len"]), np.int64)
        for i, t in enumerate(texts):
            for j, w in enumerate(str(t).split()[:DEEP["max_len"]]): a[i, j] = vocab.get(w, 1)
        return a
    def dl(df, sh):
        return DataLoader(TensorDataset(torch.tensor(enc(df[col].tolist())),
                                        torch.tensor(df["label"].map(lut).values)),
                          batch_size=DEEP["batch_size"], shuffle=sh)
    dtr, dva, dte = dl(tr, True), dl(va, False), dl(te, False)

    scores, last = [], None
    for s in seeds:
        torch.manual_seed(s); np.random.seed(s)
        m = Net(DEEP["vocab"], len(classes)).to(dev)
        opt = torch.optim.Adam(m.parameters(), lr=DEEP["lr"]); lf = nn.CrossEntropyLoss()
        best, bad, state = -1, 0, None
        for _ in range(DEEP["epochs"]):
            m.train()
            for xb, yb in dtr:
                opt.zero_grad(); lf(m(xb.to(dev)), yb.to(dev)).backward(); opt.step()
            m.eval(); P, Y = [], []
            with torch.no_grad():
                for xb, yb in dva:
                    P += m(xb.to(dev)).argmax(1).cpu().tolist(); Y += yb.tolist()
            f = macro_f1(np.array(classes)[Y], np.array(classes)[P], classes)
            if f > best: best, bad, state = f, 0, {k: v.clone() for k, v in m.state_dict().items()}
            else:
                bad += 1
                if bad >= DEEP["patience"]: break
        m.load_state_dict(state); m.eval(); P, Y = [], []
        with torch.no_grad():
            for xb, yb in dte:
                P += m(xb.to(dev)).argmax(1).cpu().tolist(); Y += yb.tolist()
        pred = np.array(classes)[P]
        scores.append(macro_f1(te["label"].values, pred, classes)); last = pred
        del m, opt; gc.collect()
        if dev == "cuda": torch.cuda.empty_cache()
    return float(np.mean(scores)), last, float(np.std(scores))

# ========================================================== transformers
def run_transformer(name, tr, va, te, col, classes=LABELS, seeds=(SEED,)):
    """Every transformer uses BUDGET. Do not change it for one model only --
    that is what made the MuRIL numbers irreconcilable across the two papers."""
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              get_linear_schedule_with_warmup)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = MODEL_CKPT[name]; lut = {c: i for i, c in enumerate(classes)}

    class DS(Dataset):
        def __init__(self, texts, labels, tok):
            self.e = tok(list(texts), truncation=True, padding="max_length",
                         max_length=BUDGET["max_len"], return_tensors="pt")
            self.y = torch.tensor(labels)
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            d = {k: v[i] for k, v in self.e.items()}; d["labels"] = self.y[i]; return d

    @torch.no_grad()
    def predict(model, loader):
        model.eval(); P = []
        for b in loader:
            b.pop("labels"); b = {k: v.to(dev) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                P += model(**b).logits.argmax(-1).cpu().tolist()
        return np.array(classes)[P]

    scores, last = [], None
    for s in seeds:
        torch.manual_seed(s); np.random.seed(s)
        tok = AutoTokenizer.from_pretrained(ckpt)
        model = AutoModelForSequenceClassification.from_pretrained(
            ckpt, num_labels=len(classes)).to(dev)
        mk = lambda d, sh: DataLoader(DS(d[col], d["label"].map(lut).values, tok),
                                      batch_size=BUDGET["batch_size"], shuffle=sh)
        dtr, dva, dte = mk(tr, True), mk(va, False), mk(te, False)
        opt = torch.optim.AdamW(model.parameters(), lr=BUDGET["lr"],
                                weight_decay=BUDGET["weight_decay"])
        total = len(dtr)*BUDGET["epochs"]
        sch = get_linear_schedule_with_warmup(opt, int(total*BUDGET["warmup_ratio"]), total)
        scaler = torch.amp.GradScaler(enabled=(dev == "cuda"))
        best, bad, state = -1.0, 0, None
        for ep in range(BUDGET["epochs"]):
            model.train()
            for b in dtr:
                b = {k: v.to(dev) for k, v in b.items()}
                opt.zero_grad()
                with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                    loss = model(**b).loss
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step()
            f = macro_f1(va["label"].values, predict(model, dva), classes)
            print(f"    epoch {ep+1:>2}/{BUDGET['epochs']}  val macro-F1 = {f:.4f}")
            if f > best:
                best, bad = f, 0
                state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= BUDGET["patience"]:
                    print(f"    early stop at epoch {ep+1}"); break
        model.load_state_dict(state)
        pred = predict(model, dte)
        scores.append(macro_f1(te["label"].values, pred, classes)); last = pred
        del model, opt, sch; gc.collect()
        if dev == "cuda": torch.cuda.empty_cache()
    return float(np.mean(scores)), last, (float(np.std(scores)) if len(scores) > 1 else None)

# =================================================================== driver
FAMILY = {**{k: "classical" for k in
             ("Logistic Regression","SVM","Naive Bayes","Random Forest","XGBoost")},
          **{k: "deep" for k in ("BiLSTM","CNN-BiLSTM")},
          **{k: "transformer" for k in MODEL_CKPT}}

def load_splits(path=f"{WORK}/splits", label_col=None):
    d = {}
    for n in ("train","val","test"):
        f = pd.read_csv(f"{path}/{n}.csv", dtype=str, keep_default_na=False)
        f.columns = [c.strip().lower() for c in f.columns]
        lc = label_col or next(c for c in ("majority","gold_label","label") if c in f.columns)
        f["label"] = f[lc].str.strip().str.lower()
        bad = int((~f["label"].isin(LABELS)).sum())
        if bad:
            print(f"  {n}: dropped {bad} rows with a missing or unexpected label")
            f = f[f["label"].isin(LABELS)].reset_index(drop=True)
        d[n] = f
    arms = [c[5:] for c in d["train"].columns if c.startswith("text_")]
    ids = {n: set(d[n]["id"]) for n in d} if "id" in d["train"].columns else None
    if ids:
        assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) \
               and not (ids["val"] & ids["test"]), "id leakage between splits"
        h = hashlib.sha256("".join(sorted(map(str, ids["test"]))).encode()).hexdigest()[:16]
        print(f"test-set id hash: {h}")
    for n in d: print(f"{n:<6}{len(d[n]):>7,}  {dict(d[n]['label'].value_counts())}")
    print("arms found:", arms)
    return d["train"], d["val"], d["test"], arms

def resolve_arms(requested, available):
    """NB1 names the transliteration arms after its XLIT_BACKEND, so the split
    files contain either itrans/keel_itrans or indicxlit/keel_indicxlit. Map
    the requested names onto whatever is actually present, and say plainly
    which engine the columns came from -- the results table must never claim
    IndicXlit when ITRANS was what ran."""
    av = set(available); out = []
    for a in requested:
        if a in av: out.append(a); continue
        alt = a.replace("indicxlit", "itrans") if "indicxlit" in a else \
              a.replace("itrans", "indicxlit")
        if alt in av:
            print(f"  arm '{a}' not in splits -> using '{alt}'")
            out.append(alt)
        else:
            print(f"  WARNING: neither '{a}' nor '{alt}' found in the splits; skipping")
    backend = "ITRANS (rule-based scheme conversion)" if any("itrans" in a for a in out) \
              else ("AI4Bharat IndicXlit (neural)" if any("indicxlit" in a for a in out)
                    else "no transliteration arm")
    print(f"  transliteration backend in these splits: {backend}")
    return out

def pred_path(model, arm, pred_dir=None):
    return f"{pred_dir or (WORK + '/preds')}/{model.replace(' ','_')}__{arm}.csv"

def find_existing_preds():
    """Predictions from earlier sessions, whether they live in /kaggle/working
    or in an attached dataset of a previous run's output."""
    seen = {}
    for d in [f"{WORK}/preds"] + glob.glob("/kaggle/input/**/preds", recursive=True):
        for f in glob.glob(f"{d}/*__*.csv"):
            seen[os.path.basename(f)] = f
    return seen

def run_grid(tr, va, te, arms, models, tag, seeds_deep=SEEDS, seeds_tf=(SEED,),
             resume=True):
    """Runs every model over every arm.

    Resumable: any model x arm whose prediction file already exists -- in this
    session or in an attached dataset from a previous run -- is scored from
    that file instead of being retrained. So the full grid can be completed
    across several Kaggle sessions by rerunning this cell unchanged.
    """
    rows, out = [], f"{WORK}/results_{tag}.csv"
    have = find_existing_preds() if resume else {}
    if have: print(f"resume: {len(have)} prediction files already on disk\n")
    for name in models:
        fam = FAMILY[name]; row = {"model": name, "family": fam}
        for arm in arms:
            col = f"text_{arm}"
            if col not in tr.columns:
                print(f"  skip {name}/{arm}: column '{col}' missing"); continue
            key = f"{name.replace(' ','_')}__{arm}.csv"
            if key in have:
                d0 = pd.read_csv(have[key], dtype=str, keep_default_na=False)
                f1 = macro_f1(d0["y_true"].values, d0["pred"].values)
                row[arm] = round(f1, 4)
                print(f"  {name} / {arm}: cached, macro-F1 {f1:.4f}")
                continue
            print(f"\n=== {name} / {arm} ===")
            if   fam == "classical":   f1, pred, sd = run_classical(name, tr, va, te, col)
            elif fam == "deep":        f1, pred, sd = run_deep(name, tr, va, te, col, seeds=seeds_deep)
            else:                      f1, pred, sd = run_transformer(name, tr, va, te, col, seeds=seeds_tf)
            row[arm] = round(f1, 4)
            if sd is not None: row[arm + "_std"] = round(sd, 4)
            pd.DataFrame({"id": te["id"] if "id" in te.columns else range(len(te)),
                          "y_true": te["label"].values, "pred": pred}) \
              .to_csv(pred_path(name, arm), index=False)
            print(f"  -> macro-F1 {f1:.4f}" + (f" +/- {sd:.4f}" if sd else ""))
            rows_so_far = rows + [row]
            pd.DataFrame(rows_so_far).to_csv(out, index=False)   # checkpoint per cell
        rows.append(row)
        pd.DataFrame(rows).to_csv(out, index=False)
    print(f"\nwrote {out}")
    return pd.DataFrame(rows)

def significance(model, baseline_arm, proposed_arm, B=10000):
    # Look in this session AND in attached datasets from earlier sessions,
    # otherwise a resumed run cannot find its own predictions.
    have = find_existing_preds()
    ka = f"{model.replace(' ','_')}__{baseline_arm}.csv"
    kb = f"{model.replace(' ','_')}__{proposed_arm}.csv"
    if ka not in have or kb not in have:
        raise FileNotFoundError(f"missing predictions: "
                                f"{[k for k in (ka, kb) if k not in have]}")
    a = pd.read_csv(have[ka]); b = pd.read_csv(have[kb])
    y = a["y_true"].values
    bs = paired_bootstrap(y, a["pred"].values, b["pred"].values, B=B)
    tt = paired_ttest(y, a["pred"].values, b["pred"].values)
    print(f"{model}: {baseline_arm} -> {proposed_arm}")
    print(f"  improvement {bs['observed']:.4f} | 95% CI [{bs['ci_low']:.4f}, {bs['ci_high']:.4f}]"
          f" | p = {bs['p_value']:.4f}")
    print(f"  paired t-test t = {tt['t']:.2f}, p = {tt['p_value']:.5f}")
    sig = bs["p_value"] < 0.05 and not (bs["ci_low"] <= 0 <= bs["ci_high"])
    print(f"\n  VERDICT: {'SIGNIFICANT' if sig else 'NOT SIGNIFICANT'} "
          f"(p={bs['p_value']:.4f}, CI {'excludes' if sig else 'includes'} zero)")
    print(f"\n--- text for the significance paragraph ---")
    if sig:
        print(f"The improvement on the full test set was {bs['observed']:.4f}, and the mean "
              f"improvement across the resamples was {bs['mean']:.4f}, with a 95% confidence "
              f"interval of [{bs['ci_low']:.4f}, {bs['ci_high']:.4f}] and p = {bs['p_value']:.4f}, "
              f"confirming that the improvement is statistically significant. A paired t-test "
              f"on the per-comment prediction correctness supported this finding "
              f"(t = {tt['t']:.2f}, p = {tt['p_value']:.5f}).")
    else:
        print(f"The difference on the full test set was {bs['observed']:.4f}, with a 95% "
              f"confidence interval of [{bs['ci_low']:.4f}, {bs['ci_high']:.4f}] and "
              f"p = {bs['p_value']:.4f}. Because the interval includes zero, the difference "
              f"is not statistically distinguishable from chance, and a paired t-test on the "
              f"per-comment prediction correctness agrees (t = {tt['t']:.2f}, "
              f"p = {tt['p_value']:.5f}). The proposed normalization therefore matches the raw "
              f"baseline rather than exceeding it, which is the claim the paper makes: the "
              f"constraint costs nothing while performing substantially fewer merges.")
    return bs, tt

def fingerprint():
    h = hashlib.sha256(open(__file__, "rb").read()).hexdigest()[:16]
    print(f"trainer v{TRAINER_VERSION}  sha256:{h}")
    print(f"  BUDGET {BUDGET}")
    print(f"  DEEP   {DEEP}")
    return h


Writing trainer.py


In [4]:
# ================================================================
#  CELL 3 — SHARED SETUP.  IDENTICAL IN BOTH NOTEBOOKS.
#  Reads the id-keyed splits written by NB1. Never re-splits.
#  Must print "trainer v1.4".
# ================================================================
import importlib, trainer
importlib.reload(trainer)
from trainer import *
import pandas as pd, numpy as np, glob, os

FP = trainer.fingerprint()
assert trainer.TRAINER_VERSION == "1.4", \
    f"stale trainer.py (v{trainer.TRAINER_VERSION}). Re-run CELL 1, then Restart session."

cands = [os.path.dirname(p) for p in
         glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert cands, ("No splits/train.csv found. Add the NB1 output as a dataset "
               "via Add Input, then rerun.")
SPLIT_DIR = cands[0]
print("splits from:", SPLIT_DIR)

tr, va, te, ARMS_AVAILABLE = load_splits(SPLIT_DIR)
print("\nSHARED SETUP DONE — the next cell is the only paper-specific one.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
splits from: /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/splits
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

SHARED SETUP DONE — the next cell is the only paper-sp

In [5]:
# ================================================================
#  PAPER 1 (LCCN) — the ONLY paper-specific cell.
#  ALL 13 arms on every model family, transformers included. The phonetic
#  arms are the high-recovery / low-accuracy tail the argument depends on,
#  so they are evaluated on transformers too rather than only on the
#  classical and deep models.
# ================================================================
WANTED = ["raw", "keel", "kope", "metaphone", "terun",
          "soundex", "soundex_guard", "indicsoundex", "indicsoundex_guard",
          "lccn_base", "lccn", "lccn_itrans", "lccn_indicxlit"]
ARMS = [a for a in WANTED if a in ARMS_AVAILABLE]
MODELS = ["Logistic Regression", "SVM", "Naive Bayes", "Random Forest", "XGBoost",
          "BiLSTM", "CNN-BiLSTM", "XLM-RoBERTa", "MuRIL"]
TAG = "paper1_lccn"
BASELINE_ARM = "raw"
PROPOSED_ARM = "lccn_indicxlit" if "lccn_indicxlit" in ARMS else "lccn_itrans"

print(f"arms ({len(ARMS)}):", ARMS)
print("proposed arm:", PROPOSED_ARM)
missing = [a for a in WANTED if a not in ARMS]
if missing: print("\nMISSING from the splits:", missing)


arms (13): ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']
proposed arm: lccn_indicxlit


In [6]:
# ================================================================
#  CELL 4 — TRANSFORMERS.  GPU REQUIRED.
#
#  ONE budget for every transformer and both papers:
#    epochs 10, lr 2e-5, batch 32, max_len 96, warmup 0.1, wd 0.01, patience 3
#  Ten epochs matters: MuRIL converges slowly on romanized input, and a short
#  budget is what made its numbers irreconcilable between the two papers.
#  Do not change the budget for one model only.
#
#  MODEL selects which transformer runs. One model per session keeps each run
#  inside Kaggle's limit. Set it, run, Save Version, then change it and repeat.
#    paper 1: XLM-RoBERTa, MuRIL
#    paper 2: XLM-RoBERTa, MuRIL, mBERT, IndicBERTv2
#
#  RESUMABLE: any model x arm whose prediction file already exists is scored
#  from disk instead of retrained. To continue, add this notebook's own output
#  as an input to itself and rerun.
#
#  Cost: roughly 45-60 min per arm. Paper 2 is 9 arms per model, paper 1 is 13.
#  Expect one model per session, sometimes two sessions for paper 1.
# ================================================================
import torch
assert torch.cuda.is_available(), "GPU is off — Settings > Accelerator > GPU"
print("gpu:", torch.cuda.get_device_name(0))

MODEL  = "XLM-RoBERTa"        # <- change this between sessions
SEEDS_TF = (42,)              # (42, 43, 44) only for the arms that carry the claim

assert MODEL in MODELS, f"{MODEL} not in this paper's MODELS: {MODELS}"
print(f"\nmodel={MODEL}  checkpoint={MODEL_CKPT[MODEL]}")
print(f"budget={BUDGET}")
print(f"arms={ARMS}  ({len(ARMS)} runs x {len(SEEDS_TF)} seed(s))\n")

res = run_grid(tr, va, te, ARMS, [MODEL],
               tag=f"{TAG}_transformer_{MODEL.replace('-','').replace(' ','')}",
               seeds_tf=SEEDS_TF, resume=True)
print("\n=== macro-F1 ===")
print(res.to_string(index=False))

cols = [a for a in ARMS if a in res.columns]
r = res.iloc[0]
order = sorted(cols, key=lambda c: r[c] if pd.notna(r[c]) else -1, reverse=True)
print(f"\n{MODEL} ranked by macro-F1:")
for a in order:
    print(f"  {a:<22}{r[a]:.4f}   {r[a]-r['raw']:+.4f} vs raw")
print(f"\nprediction files on disk: {len(find_existing_preds())}")


gpu: Tesla T4

model=XLM-RoBERTa  checkpoint=xlm-roberta-base
budget={'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
arms=['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']  (13 runs x 1 seed(s))


=== XLM-RoBERTa / raw ===


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7484
    epoch  2/10  val macro-F1 = 0.7969
    epoch  3/10  val macro-F1 = 0.7964
    epoch  4/10  val macro-F1 = 0.8175
    epoch  5/10  val macro-F1 = 0.8170
    epoch  6/10  val macro-F1 = 0.8196
    epoch  7/10  val macro-F1 = 0.8228
    epoch  8/10  val macro-F1 = 0.8244
    epoch  9/10  val macro-F1 = 0.8236
    epoch 10/10  val macro-F1 = 0.8194
  -> macro-F1 0.8051

=== XLM-RoBERTa / lccn_indicxlit ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  1/10  val macro-F1 = 0.7551
    epoch  2/10  val macro-F1 = 0.8068
    epoch  3/10  val macro-F1 = 0.8016
    epoch  4/10  val macro-F1 = 0.8210
    epoch  5/10  val macro-F1 = 0.8198
    epoch  6/10  val macro-F1 = 0.8222
    epoch  7/10  val macro-F1 = 0.8173
    epoch  8/10  val macro-F1 = 0.8127
    epoch  9/10  val macro-F1 = 0.8138
    early stop at epoch 9
  -> macro-F1 0.8130

wrote /kaggle/working/results_paper1_lccn_transformer_XLMRoBERTa.csv

=== macro-F1 ===
      model      family    raw   keel  kope  metaphone  terun  soundex  soundex_guard  indicsoundex  indicsoundex_guard  lccn_base   lccn  lccn_itrans  lccn_indicxlit
XLM-RoBERTa transformer 0.8058 0.8004 0.809     0.7916 0.8041   0.7786         0.7792        0.7773              0.7861     0.7991 0.7991       0.8051           0.813

XLM-RoBERTa ranked by macro-F1:
  lccn_indicxlit        0.8130   +0.0072 vs raw
  kope                  0.8090   +0.0032 vs raw
  raw                   0.8058   +0.0000 vs raw
  l

In [7]:
# ================================================================
#  CELL 6 — ASSEMBLE THE TABLE AND TEST SIGNIFICANCE.  IDENTICAL IN BOTH.
#  Run after the stages you need have completed.
# ================================================================
import glob
parts = [pd.read_csv(f) for f in sorted(glob.glob(f"{WORK}/results_{TAG}_*.csv"))]
if not parts:
    print("no results yet — run CELL 5 first")
else:
    full = pd.concat(parts, ignore_index=True)
    full.to_csv(f"{WORK}/results_{TAG}.csv", index=False)
    print("=== FULL RESULTS TABLE (macro-F1) ===")
    print(full.to_string(index=False))

    cols = [a for a in ARMS if a in full.columns]
    full["best_arm"] = full.apply(
        lambda r: max(cols, key=lambda c: r[c] if pd.notna(r[c]) else -1), axis=1)
    print("\nbest arm per model:")
    print(full[["model", "best_arm"]].to_string(index=False))
    n = int((full["best_arm"] == PROPOSED_ARM).sum())
    print(f"\n{PROPOSED_ARM} is best for {n} of {len(full)} models")

    # per-class F1 for the proposed arm, for the error-analysis section
    f = f"{WORK}/preds/XLM-RoBERTa__{PROPOSED_ARM}.csv"
    if os.path.exists(f):
        d0 = pd.read_csv(f)
        print("\nper-class P / R / F1 (XLM-RoBERTa, proposed arm):")
        for c, (p_, r_, f1_) in per_class_prf(d0.y_true.values, d0.pred.values, LABELS).items():
            print(f"  {c:<10} P={p_:.4f}  R={r_:.4f}  F1={f1_:.4f}")

for m in ["XLM-RoBERTa", "Logistic Regression"]:
    try:
        print("\n" + "="*62)
        significance(m, BASELINE_ARM, PROPOSED_ARM)
    except FileNotFoundError:
        print(f"{m}: predictions for {BASELINE_ARM}/{PROPOSED_ARM} not found yet")

print(f"\ntrainer {FP} | tag {TAG}")


=== FULL RESULTS TABLE (macro-F1) ===
      model      family    raw   keel  kope  metaphone  terun  soundex  soundex_guard  indicsoundex  indicsoundex_guard  lccn_base   lccn  lccn_itrans  lccn_indicxlit
XLM-RoBERTa transformer 0.8058 0.8004 0.809     0.7916 0.8041   0.7786         0.7792        0.7773              0.7861     0.7991 0.7991       0.8051           0.813

best arm per model:
      model       best_arm
XLM-RoBERTa lccn_indicxlit

lccn_indicxlit is best for 1 of 1 models

per-class P / R / F1 (XLM-RoBERTa, proposed arm):
  anger      P=0.7758  R=0.6507  F1=0.7078
  disgust    P=0.6920  R=0.7713  F1=0.7295
  happy      P=0.8909  R=0.9093  F1=0.9000
  sad        P=0.9032  R=0.9267  F1=0.9148

XLM-RoBERTa: raw -> lccn_indicxlit
  improvement 0.0072 | 95% CI [-0.0014, 0.0158] | p = 0.0986
  paired t-test t = 1.87, p = 0.06210

  VERDICT: NOT SIGNIFICANT (p=0.0986, CI includes zero)

--- text for the significance paragraph ---
The difference on the full test set was 0.0072, wit